> ⚠️ **Legacy notebook — superseded by `test_chat_mode.ipynb`**
>
> This notebook tested a Python-level orchestration step that no longer exists
> after the migration to the OpenAI Agents SDK (commit `762f7a9`, April 2026).
> Under the A1 design, the orchestrator agent runs as a single `Runner.run` and
> the legacy classes / functions referenced below have been deleted.
>
> **For the current workflow** — including intermediate results (questions
> after `relevance_check`, sub-questions to specialists, general_specialist's
> review, etc.) — see [`test_chat_mode.ipynb`](test_chat_mode.ipynb), which
> exercises the production pipeline and exposes every stage via
> `result.new_items`.
>
> **Why this notebook is kept**: as historical reference and as a starting
> template if you want to write a focused iteration notebook for a single
> agent factory (`build_specialist_agent`, `build_report_agent`,
> `build_general_specialist`, `build_orchestrator_agent`) under the new SDK.


# Data Query — Iteration Notebook

Polish a single specialist's tool-calling loop against the data gateway.

**Edit surfaces:**
- Non-engineer: `skills/workflow/data_query.md`, `skills/workflow/data_catalog.md`, `skills/domain/<specialist>.md`, `config/data_profiles/*.yaml`.
- Engineer: `tools/data_tools.py`, `agents/base_agent.py`.

**Workflow:** edit a file above → Run All → read Cell 5's tool-call trace + specialist output → repeat. Cell 6's raw JSON is the regression signal.

**Switching specialists:** set `fixture["specialist"]` to any domain under `skills/domain/` (e.g., `bureau`, `wcc`, `modeling`, `crossbu`, `customer_rel`, `capacity_afford`, `spend_payments`).

In [ ]:
# ═══════════════ KNOBS ═══════════════
FIXTURE = "basic_case"
REGENERATE = False
MODEL = "gpt-4.1"
# ═════════════════════════════════════

import json
import os
import sys
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.session_registry import SessionRegistry
from config.pillar_loader import PillarLoader
from datalayer.catalog import DataCatalog
from datalayer.gateway import LocalDataGateway
from datalayer.generator import DataGenerator
from llm.firewall_stack import FirewallStack
from llm.factory import build_llm
from logger.event_logger import EventLogger
from skills.domain.loader import load_domain_skill
from tools.data_tools import init_tools

FIXTURE_PATH = PROJECT_ROOT / "notebooks" / "fixtures" / "data_query" / f"{FIXTURE}.json"
print(f"Fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")

In [ ]:
logger = EventLogger(session_id="polish-data-query")
firewall = FirewallStack(logger=logger)
llm = build_llm(MODEL, firewall)

_DATA_TABLES_DIR = PROJECT_ROOT / "data_tables"
csv_gateway = LocalDataGateway.from_case_folders(str(_DATA_TABLES_DIR))
if csv_gateway.list_case_ids():
    gateway = csv_gateway
else:
    gen = DataGenerator(
        profile_dir=str(PROJECT_ROOT / "config" / "data_profiles"),
        seed=42, cases=50,
    )
    gen.load_profiles()
    tables_raw = gen.generate_all()
    gateway = LocalDataGateway.from_generated(tables_raw)

catalog = DataCatalog(profile_dir=str(PROJECT_ROOT / "config" / "data_profiles"))
init_tools(gateway, catalog, logger=logger)
registry = SessionRegistry()
pillar_loader = PillarLoader(pillar_dir=str(PROJECT_ROOT / "config" / "pillars"))
print(f"Available case IDs (first 5): {gateway.list_case_ids()[:5]}")

In [ ]:
if REGENERATE:
    current = {
        "question": "Summarise this customer's recent spend and payment behaviour.",
        "pillar": "credit_risk",
        "case_id": gateway.list_case_ids()[0],
        "specialist": "spend_payments",
        "notes": f"Regenerated from case {gateway.list_case_ids()[0]}, specialist spend_payments.",
    }
    FIXTURE_PATH.write_text(json.dumps(current, indent=2) + "\n")
    fixture = current
    print(f"Wrote fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")
else:
    fixture = json.loads(FIXTURE_PATH.read_text())
    print(f"Loaded fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")

# Pick a usable case_id (same pattern as Task 2).
available = gateway.list_case_ids()
case_id = fixture["case_id"] if fixture["case_id"] in available else available[0]
if case_id != fixture["case_id"]:
    print(f"  (fixture case '{fixture['case_id']}' not available; using '{case_id}')")
gateway.set_case(case_id)

pillar_yaml = pillar_loader.load(fixture["pillar"]) or {}

skill = load_domain_skill(fixture["specialist"])
assert skill is not None, f"Unknown specialist {fixture['specialist']!r} — check skills/domain/."

print(f"Pillar: {fixture['pillar']} | Case: {case_id} | Specialist: {fixture['specialist']}")
print(f"Question: {fixture['question']}")

In [ ]:
from IPython.display import Markdown, display

agent = registry.get_or_create(
    domain=fixture["specialist"],
    pillar=fixture["pillar"],
    domain_skill=skill,
    pillar_yaml=pillar_yaml,
    llm=llm,
    logger=logger,
)

# Mark where the trace starts so we can scrape only this run's events.
_TRACE_START = sum(1 for _ in open(logger._file_path)) if os.path.exists(logger._file_path) else 0

output = await agent.run(
    fixture["question"], mode="chat", root_question=fixture["question"],
)

# Scrape the logger's JSONL for tool-related events from this run onward.
tool_events = []
if os.path.exists(logger._file_path):
    with open(logger._file_path) as f:
        for i, line in enumerate(f):
            if i < _TRACE_START:
                continue
            evt = json.loads(line)
            if evt.get("event") in {"data_request", "data_response", "tool_call", "tool_result", "query_table", "get_table_schema", "list_available_tables"}:
                tool_events.append(evt)

lines = [f"### Question\n\n{fixture['question']}\n"]
lines.append(f"**Specialist:** `{fixture['specialist']}`  |  **Tables declared:** {', '.join(skill.data_hints) or '(none)'}\n")
lines.append(f"**Tool-related events ({len(tool_events)}):**")
if not tool_events:
    lines.append("- _(none captured — the logger may emit different event names in this codebase version)_")
for evt in tool_events:
    evt_type = evt.get("event", "?")
    payload = {k: v for k, v in evt.items() if k not in {"timestamp", "session_id", "trace_id", "event"}}
    lines.append(f"- `{evt_type}` — {json.dumps(payload)[:300]}")

lines.append(f"\n**Findings:**\n\n{output.findings}\n")
if output.evidence:
    lines.append("**Evidence:**")
    for e in output.evidence:
        lines.append(f"- {e}")
if output.implications:
    lines.append("\n**Implications:**")
    for i in output.implications:
        lines.append(f"- {i}")
if output.data_gaps:
    lines.append("\n**Data gaps:**")
    for g in output.data_gaps:
        lines.append(f"- {g}")
display(Markdown("\n".join(lines)))

In [ ]:
print(json.dumps(output.model_dump(), indent=2))